# GraphRAG: Community Detection and the Modularity of Knowledge

*The final topic in the Information Theory of RAG layer — from retrieval as a **search** (a path) to retrieval as a **partition** (the global structure).*

Multi-hop retrieval closed on the line *"the relation 'retrievable-from' has drawn a graph over the whole corpus,
and the question shifts from finding a single path to understanding that graph's global structure — its dense
communities of mutually-retrievable filings."* This notebook makes that precise, walking the five movements.
Every number is owned by the tested reference module `graphrag_community_detection.py`, which this notebook
imports — it never redefines the math.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd()))
import numpy as np
import graphrag_community_detection as G

g = G.graphrag_corpus()
A, sector = g["A"], g["sector"]
print(f"entity graph: {A.shape[0]} companies, {G.GR_N_SECTORS} planted sectors (the communities), "
      f"{int((A > 0).sum() // 2)} weighted co-occurrence edges")

## 1 · The entity graph and modularity

GraphRAG builds a graph whose nodes are entities (companies) and whose edges are co-occurrence — here the sharpened
cosine of the shared finance vMF geometry, so **sectors are the planted communities**. Newman–Girvan modularity
scores a partition against the degree-preserving (configuration-model) null,
$Q = \frac{1}{2m}\sum_{ij}\left[A_{ij}-\frac{k_ik_j}{2m}\right]\delta(c_i,c_j)$: the excess of within-community
edge weight over chance. The planted sectors score far above a random partition.

In [ ]:
Q_planted = G.modularity(A, sector)
rng = np.random.default_rng(0)
Q_random = float(np.mean([G.modularity(A, rng.permutation(sector)) for _ in range(50)]))
print(f"Q(planted sectors) = {Q_planted:.3f}   >>   Q(random partition) = {Q_random:.3f}")

## 2 · The spectral relaxation (a theorem)

For a bipartition $s\in\{\pm1\}^n$, the modularity matrix $B=A-\mathbf{k}\mathbf{k}^\top/(2m)$ has rows summing to
zero, so $Q=\frac{1}{4m}s^\top B s$. Relaxing $s$ to the sphere, $Q$ is maximized by $B$'s **leading (largest
algebraic) eigenvector**, rounded by sign — Newman's spectral method. On a small two-clique graph it matches the
brute-force modularity-optimal bipartition exactly; a single clique has $\lambda_1(B)\le 0$ and is **indivisible**.

In [ ]:
# small two-clique graph -> spectral == brute force
Atoy = np.zeros((8, 8))
for clq in ([0, 1, 2, 3], [4, 5, 6, 7]):
    for a in clq:
        for b in clq:
            if a < b:
                Atoy[a, b] = Atoy[b, a] = 1.0
Atoy[3, 4] = Atoy[4, 3] = 1.0
sp, lam1 = G.spectral_bipartition(Atoy)
bf, Qbf = G.brute_modularity_argmax(Atoy)
print(f"spectral sign vector = {sp.tolist()}")
print(f"brute-force argmax    = {bf.tolist()}   (Q = {Qbf:.3f}, lambda_1(B) = {lam1:.3f} > 0: divisible)")
_, lam_clique = G.spectral_bipartition(np.ones((6, 6)) - np.eye(6))
print(f"a clique K6: lambda_1(B) = {lam_clique:.2e} <= 0  ->  indivisible")

## 3 · The resolution limit (the load-bearing rigorFlag)

Fortunato–Barthélemy: modularity cannot resolve communities smaller than a scale set by the **whole** graph,
$\sim\sqrt{2m}$. On a large ring of cliques the modularity-optimal partition **merges adjacent cliques** even
though each clique is a genuine community — the $\sqrt{2m}$ normalization makes a small clique invisible against a
large background. The resolution parameter $\gamma$ only *moves* the scale; on a small ring the cliques are kept
separate (the contrast).

In [ ]:
for nc in (3, G.RING_LARGE_NC):
    Ar, true = G.ring_of_cliques(nc, G.RING_CLIQUE)
    Qs = G.modularity(Ar, true)
    Qp = G.modularity(Ar, G.paired_clique_labels(nc, G.RING_CLIQUE)) if nc % 2 == 0 else float("nan")
    m2 = G.total_weight(Ar)
    print(f"{nc:>2} cliques (sqrt(2m)={np.sqrt(m2):.1f}): Q(singles)={Qs:.3f}, Q(pairs)={Qp:.3f}")
A30, _ = G.ring_of_cliques(G.RING_LARGE_NC, G.RING_CLIQUE)
lab30, _ = G.louvain(A30)
print(f"Louvain on the 30-clique ring finds {G.relabel_consecutive(lab30)[1]} communities (< 30: it MERGED)")

## 4 · The stochastic block model and the detectability transition (the deep payload)

The SBM is the generative model: $n$ nodes in $q$ blocks, an edge w.p. $p_{\text{in}}$ within and $p_{\text{out}}$
across. For the symmetric two-block sparse SBM the **Kesten–Stigum threshold** (Decelle et al. 2011, proven by
Mossel–Neeman–Sly / Massoulié) says a partition correlated with the planted one is recoverable **iff**
$(c_{\text{in}}-c_{\text{out}})^2 > 2(c_{\text{in}}+c_{\text{out}})$. Below it, *no algorithm* beats a coin flip —
an information-theoretic converse, the analogue of Fano in the noisy-channel topic. We sample on both sides.

In [ ]:
for name, (ci, co) in (("above", G.SBM_ABOVE), ("below", G.SBM_BELOW)):
    ov = G.sbm_point_overlap(ci, co, G.SBM_N, G.GR_SEED)
    print(f"{name:>5} threshold (c_in={ci}, c_out={co}, SNR={G.ks_snr(ci, co):.2f}): "
          f"recovery overlap = {ov:.3f}   ({'recovered' if ov > 0.5 else 'at chance'})")

## 5 · GraphRAG: Louvain, Leiden, and the modularity of knowledge

Modularity maximization is NP-hard (Brandes 2008), so **Louvain** (local-moving + aggregation) and **Leiden**
(adding a refinement phase that *guarantees* internally-connected communities) are heuristics. A disconnected
community can be a Louvain **local optimum** that local-moving cannot repair; Leiden's refinement forbids it.
GraphRAG (Edge et al. 2024) runs hierarchical Leiden on the entity graph and map-reduces per-community summaries
to answer global/sensemaking queries — a thematic decomposition that exists only when $Q$ is high and the SBM
signal clears Kesten–Stigum.

In [ ]:
lab = G.leiden(A)
print(f"Leiden recovers the planted sectors: overlap = {G.overlap_multi(sector, lab):.3f}")
Aw, stuck = G.louvain_disconnected_witness()
print(f"a disconnected community is a Louvain local optimum: connectivity {G.community_is_connected(Aw, stuck)}")
print(f"Leiden's refinement repairs it: {G.community_is_connected(Aw, G.leiden_refine(Aw, stuck))}")

## Verification

Every pedagogical claim above is an assertion in the reference module; `viz_constants()` prints the numbers the
interactive lab mirrors to the decimal.

In [ ]:
G._run_all()

In [ ]:
G.viz_constants()